## **Install gensim & PyTorch**

In [1]:
#Install gensim
!pip install --upgrade gensim
#Install PyTorch (gpu version)
!pip install torch torchvision

     |████████████████████████████████| 24.2MB 124kB/s 
  Found existing installation: gensim 3.6.0
    Uninstalling gensim-3.6.0:
      Successfully uninstalled gensim-3.6.0


In [2]:
#Verify Pytorch installation
import torch
print(torch.__version__)

1.5.0+cu101


In [4]:
!nvidia-smi

Mon May 11 20:02:23 2020       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 440.82       Driver Version: 418.67       CUDA Version: 10.1     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|===============================+======================+======================|
|   0  Tesla P100-PCIE...  Off  | 00000000:00:04.0 Off |                    0 |
| N/A   43C    P0    27W / 250W |      0MiB / 16280MiB |      0%      Default |
+-------------------------------+----------------------+----------------------+
                                                                               
+-----------------------------------------------------------------------------+
| Processes:                                                       GPU Memory |
|  GPU  

## **Download and process 20newsgroups dataset**

In [5]:
# Fetch data
from sklearn.datasets import fetch_20newsgroups
twenty_train = fetch_20newsgroups(subset='train')

# Split into training and dev
from sklearn.model_selection import train_test_split  
X_train, X_val, y_train, y_val = train_test_split(twenty_train.data, twenty_train.target, test_size=0.3, random_state=12547392)

# Tokenize with spacy
import spacy
nlp = spacy.load('en_core_web_sm',disable=["tagger", "parser","ner"])
from spacy.lang.en.stop_words import STOP_WORDS
nlp.add_pipe(nlp.create_pipe('sentencizer')) 

X_train_tokenized = []
for idx in range(len(X_train)):
  doc = nlp(X_train[idx])
  tokens = []
  for sent in doc.sents:
    for tok in sent:
      if '\n' in tok.text or "\t" in tok.text or "--" in tok.text or "*" in tok.text or tok.text.lower() in STOP_WORDS:
        continue
      if tok.text.strip():  
        tokens.append(tok.text.replace('"',"'").strip())
  X_train_tokenized.append(tokens)

X_val_tokenized = []
for idx in range(len(X_val)):
  doc = nlp(X_val[idx])
  tokens = []
  for sent in doc.sents:
    for tok in sent:
      if '\n' in tok.text or "\t" in tok.text or "--" in tok.text or "*" in tok.text or tok.text.lower() in STOP_WORDS:
        continue
      if tok.text.strip():
        tokens.append(tok.text.replace('"',"'").strip())
  X_val_tokenized.append(tokens)

## **Download, unzip & load fasttext word embeddings**

In [6]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz

--2020-05-11 20:08:08--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 104.22.75.142, 104.22.74.142, 2606:4700:10::6816:4a8e, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|104.22.75.142|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4503593528 (4.2G) [application/octet-stream]
Saving to: ‘cc.en.300.bin.gz’

cc.en.300.bin.gz     32%[=====>              ]   1.36G  21.3MB/s    in 76s     

2020-05-11 20:09:25 (18.4 MB/s) - Read error at byte 1459004752/4503593528 (Connection reset by peer). Retrying.

--2020-05-11 20:09:26--  (try: 2)  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|104.22.75.142|:443... connected.
HTTP request sent, awaiting response... 206 Partial Content
Length: 4503593528 (4.2G), 3044588776 (2.8G) remaining [application/octet-stream]
Saving to: ‘cc.en.30

In [0]:
!gzip -d cc.en.300.bin.gz

In [0]:
from gensim.models.wrappers import FastText

fasttext = FastText.load_fasttext_format('cc.en.300.bin')

## **Calculate centroids**

In [0]:
#Calculate centroid function

import numpy as np
from spacy.lang.en.stop_words import STOP_WORDS


def text_centroid(text, model):
    
    text_vec =[]
    counter = 0
    for word in text:
        if word in STOP_WORDS:
          continue         
        try:
            if counter == 0:
                text_vec = model[word.lower()]
            else:
                text_vec = np.add(text_vec, model[word.lower()])
            counter+=1
        except:
            pass
    
    return np.asarray(text_vec) / counter

In [0]:
# Calculate centroids for train and test documents

import numpy as np 

X_train_centroids = []
for sentence in X_train_tokenized:
    X_train_centroids.append(text_centroid(sentence,fasttext))   
X_train_centroids = np.stack(X_train_centroids, axis=0)

X_val_centroids = []
for sentence in X_val_tokenized:
    X_val_centroids.append(text_centroid(sentence,fasttext))   
X_val_centroids = np.stack(X_val_centroids, axis=0)

## **Dataloader class**

In [0]:
import torch
import numpy as np
import math
import random

class TextClassDataLoader(object):

    def __init__(self,data,labels=None,batch_size=32,predict_flag=0,train=0):
        """

        Args:
            data: numpy array with text centroid embedding
            labels: numpy array with text labels
            batch_size:
        """

        self.batch_size = batch_size
        self.predict_flag = predict_flag
        self.train = train
        print("Train flag: ",self.train)
        print("Predict flag: ",self.predict_flag)
        
        self.data = torch.Tensor(data)
        if type(labels) == np.ndarray:
            self.labels = torch.LongTensor(labels)
        
        # for batch
        self.n_samples = self.data.size()[0]
        self.n_batches = math.ceil(self.n_samples / self.batch_size)
        self.index = 0
        self.batch_index = 0
        self.indices = np.arange(self.n_samples)
        if self.train:
            self._shuffle_indices()

        self.report()

    def _shuffle_indices(self):
        self.indices = np.random.permutation(self.n_samples)
        self.index = 0
        self.batch_index = 0

  
    def _create_batch(self):

        batch = []
        n = 0
        while ((n < self.batch_size) and (self.index < self.n_samples)):
            _index = self.indices[self.index]
            batch.append(_index)
            self.index += 1
            n += 1
        self.batch_index += 1

        #Fix for the extreme case that last batch has size == 1. Append the last sample to the
        #previous batch
        if (self.index+1 == self.n_samples):
            _index = self.indices[self.index]
            batch.append(_index)
            self.index += 1
            self.batch_index += 1

        batch_indices = torch.Tensor(batch).long()
        
        input_tensor = torch.index_select(self.data,0,batch_indices)
        
        if type(self.labels) == torch.Tensor:
            target_tensor = torch.index_select(self.labels,0,batch_indices)
        else:
            target_tensor = None
        
        return input_tensor, target_tensor

    def __len__(self):
        return self.n_batches

    def __iter__(self):

        if self.train:
            self._shuffle_indices()
        else:
            self.index = 0
            self.batch_index = 0

        for i in range(self.n_batches):
            if self.batch_index == self.n_batches:
                raise StopIteration()
            yield self._create_batch()

 
    def report(self):
        print('# samples: {}'.format(self.n_samples))
        print('# batches: {} (batch_size = {})'.format(self.n_batches, self.batch_size))


## **MLP class**

In [0]:
#Define MLP model

import torch
import numpy as np
import torch.nn as nn
import math

class MLP(nn.Module):
    
    def __init__(self, input_size, num_classes, hidden_layers, hidden_sizes, dropout):
        
        """

        Args:
            input_size: dim of input tensor
            num_classes: dim of output (# classes)
            hidden_layers: number of hidden layers
            hidden_sizes: dim of hidden layers (list)
            dropout: dropout probability
            
        """
        
        super(MLP, self).__init__()
        
        self.input_size = input_size
        self.classes = num_classes
        self.hidden_layers = hidden_layers
        self.hidden_sizes = hidden_sizes
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.ReLU()
        
        if (self.hidden_layers != len(self.hidden_sizes)):
            raise ValueError("Number of hidden layers does not match with list of hidden layers dims")
        if self.hidden_layers < 1:
             raise ValueError("MLP should have at least 1 hidden layer")
        for hidden_dim in self.hidden_sizes:
            if hidden_dim < 1:
                raise ValueError("Hidden layers should have at least 1 hidden unit")
        
        for layer_index in range(self.hidden_layers):
            
            if layer_index == 0:
                hidden_layer = nn.Linear(self.input_size, self.hidden_sizes[layer_index])
            else:
                hidden_layer = nn.Linear(self.hidden_sizes[layer_index -1],self.hidden_sizes[layer_index])
            
            self.add_module('hidden_layer_{}'.format(layer_index), hidden_layer)
        
        self.out =  nn.Linear(self.hidden_sizes[-1],self.classes)
        
    
    def forward(self, input_tensor):
        
        input_t = input_tensor
        for layer_index in range(self.hidden_layers):
            hidden = getattr(self, 'hidden_layer_{}'.format(layer_index))
            output = self.activation(hidden(input_t))
            input_t = self.dropout(output)
        
        out = self.out(input_t)
        
        return out

## **Create train & validation dataloaders**

In [24]:
batch_size = 512
train_loader = TextClassDataLoader(X_train_centroids,y_train,batch_size,predict_flag=0,train=1)
print("____________")
val_loader = TextClassDataLoader(X_val_centroids,y_val,batch_size,predict_flag=0,train=0)

Train flag:  1
Predict flag:  0
# samples: 7919
# batches: 16 (batch_size = 512)
____________
Train flag:  0
Predict flag:  0
# samples: 3395
# batches: 7 (batch_size = 512)


## **Create and train an MLP model for text classification**

In [17]:
# Instanciate an MLP model
model = MLP(X_train_centroids.shape[1],len(twenty_train.target_names),3,[1024,1024,512],dropout=0.5)
print(model)

MLP(
  (dropout): Dropout(p=0.5, inplace=False)
  (activation): ReLU()
  (hidden_layer_0): Linear(in_features=300, out_features=1024, bias=True)
  (hidden_layer_1): Linear(in_features=1024, out_features=1024, bias=True)
  (hidden_layer_2): Linear(in_features=1024, out_features=512, bias=True)
  (out): Linear(in_features=512, out_features=20, bias=True)
)


In [42]:
from tqdm.notebook import tqdm
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score


# If gpu availiable use it, otherwise use CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)
#define optimizer and loss
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
criterion = nn.CrossEntropyLoss()

#Train for 100 epochs and validate at the end of each epoch
#Keep the model with the best validation f1-score
epochs = 100
highest_val_f1 = 0
for idx in tqdm(range(epochs),desc="Epoch"):
    epoch = idx+1
    #Switch to train mode
    model.train()
    for ti, (input_t, target_t) in enumerate(train_loader):

        input_t = input_t.to(device)
        target_t = target_t.to(device)
        output = model(input_t)
        loss = criterion(output,target_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    #Switch to eval mode
    model.eval()
    val_loss = []
    val_targets = []
    val_outputs = []
    for vi, (input_t, target_t) in enumerate(val_loader):
        input_t = input_t.to(device)
        target_t = target_t.to(device)
        output = model(input_t)
        val_outputs.append(output)
        val_targets.append(target_t)
        val_loss.append(criterion(output,target_t).detach().cpu().numpy())
    val_outputs = torch.cat(val_outputs,0)
    val_targets = torch.cat(val_targets,0)
    f1 = f1_score(val_targets.detach().cpu().numpy(), np.argmax(val_outputs.detach().cpu().numpy(),axis=1),average="weighted")
    precision = precision_score(val_targets.detach().cpu().numpy(), np.argmax(val_outputs.detach().cpu().numpy(),axis=1),average="weighted")
    recal = recall_score(val_targets.detach().cpu().numpy(), np.argmax(val_outputs.detach().cpu().numpy(),axis=1),average="weighted") 
    acc = accuracy_score(val_targets.detach().cpu().numpy(), np.argmax(val_outputs.detach().cpu().numpy(),axis=1)) 
    print("val loss: %.4f , val f1: %.3f, val precision: %.3f, val recall: %.3f, val accuracy: %.3f" %((sum(val_loss)/len(val_loss)),f1,precision,recal,acc))
    if f1 > highest_val_f1:
        print("Save model....")
        torch.save({'model_state_dict': model.state_dict()}, "pytorch_model_2.bin")
        highest_val_f1 = f1

val loss: 1.2848 , val f1: 0.775, val precision: 0.779, val recall: 0.776, val accuracy: 0.776
Save model....
val loss: 1.2744 , val f1: 0.780, val precision: 0.785, val recall: 0.780, val accuracy: 0.780
Save model....
val loss: 1.2465 , val f1: 0.782, val precision: 0.784, val recall: 0.782, val accuracy: 0.782
Save model....
val loss: 1.2386 , val f1: 0.788, val precision: 0.791, val recall: 0.787, val accuracy: 0.787
Save model....
val loss: 1.2686 , val f1: 0.787, val precision: 0.791, val recall: 0.786, val accuracy: 0.786
val loss: 1.2736 , val f1: 0.779, val precision: 0.784, val recall: 0.779, val accuracy: 0.779
val loss: 1.2534 , val f1: 0.782, val precision: 0.786, val recall: 0.781, val accuracy: 0.781
val loss: 1.2293 , val f1: 0.781, val precision: 0.783, val recall: 0.781, val accuracy: 0.781
val loss: 1.2554 , val f1: 0.781, val precision: 0.784, val recall: 0.781, val accuracy: 0.781
val loss: 1.2611 , val f1: 0.776, val precision: 0.781, val recall: 0.775, val accura

## **Evaluate performance of centroids MLP model on validation set**



In [46]:
from sklearn.metrics import classification_report

#Define loss function
criterion = criterion = nn.CrossEntropyLoss()

#Create test dataloader
test_loader = TextClassDataLoader(X_val_centroids,y_val,batch_size=128,predict_flag=0,train=0)

#Create model instance using identical parameters with the already trained model
model_eval = MLP(X_val_centroids.shape[1],len(twenty_train.target_names),3,[1024,1024,512],dropout=0)

#Load pre-trained model
print("===> Loading pretrained model ...")
checkpoint = torch.load("pytorch_model_2.bin", map_location="cpu")
model.load_state_dict(checkpoint['model_state_dict'])

#Switch to evaluation mode
model.eval()
val_targets = []
val_outputs = []

for i, (input_t, target_t) in enumerate(val_loader):
    input_t = input_t.to(device)
    target_t = target_t.to(device)
    output = model(input_t)
    val_outputs.append(output)
    val_targets.append(target_t)

val_outputs = torch.cat(val_outputs,0)
val_targets = torch.cat(val_targets,0)
print(classification_report(val_targets.detach().cpu().numpy(), np.argmax(val_outputs.detach().cpu().numpy(),axis=1), target_names=twenty_train.target_names))

Train flag:  0
Predict flag:  0
# samples: 3395
# batches: 27 (batch_size = 128)
===> Loading pretrained model ...
                          precision    recall  f1-score   support

             alt.atheism       0.78      0.68      0.73       160
           comp.graphics       0.57      0.68      0.62       165
 comp.os.ms-windows.misc       0.74      0.69      0.71       189
comp.sys.ibm.pc.hardware       0.66      0.68      0.67       168
   comp.sys.mac.hardware       0.71      0.63      0.66       182
          comp.windows.x       0.83      0.83      0.83       168
            misc.forsale       0.76      0.73      0.75       182
               rec.autos       0.86      0.77      0.81       181
         rec.motorcycles       0.86      0.83      0.85       184
      rec.sport.baseball       0.88      0.90      0.89       169
        rec.sport.hockey       0.90      0.94      0.92       175
               sci.crypt       0.87      0.85      0.86       177
         sci.electronics  